# Laya fine-tune: train, compare, report (one notebook, press **Run all**)

What it does, in order: checks the GPU, mounts Drive, installs the pinned libraries, downloads the untouched English checkpoint, scores the **base model** on the dev and test rows, trains two variants (soft-ce and rlcd), scores both, picks the better one on **dev**, saves everything to Drive, and prints a short summary to paste back.

Expect roughly 30 to 40 minutes on an L4. The data is embedded in this notebook (nothing to upload).

**Read this before trusting any number:** every train, dev and test row was written *and labelled by Claude*. No row is human-labelled and the test set is not independent. The test rows use industries, personas and wording that the training rows never contain, so the test gap is a fair sign of generalisation, but the scores say "good on Claude-written synthetic calls", not "good on real calls".

If a cell fails, the error is printed under it. Paste that to Claude. After a kernel restart: run cell 1, then continue from the cell that failed.

Nothing here modifies the original checkpoint: training reads a downloaded copy and writes new folders.

In [ ]:
# CELL 1: settings + GPU check.
# EXPECT: "NVIDIA L4, 23034 MiB" (about 23 GB), "cuda True", then the run id.
import datetime
import hashlib
import json
import os
import queue
import re
import shutil
import subprocess
import sys
import threading
import time

import torch

BASE_REPO = "convaiinnovations/laya"
BASE_REV = "7b928d828b7b0e022f929d9bd2e44165aa270148"      # the snapshot used locally (laya-test/hf_home)
LAYA_COMMIT = "3cf26cbcb18725dbc2d127bb8bb2c4c43243ae63"   # laya v0.4.0
LOCAL_BASE_SHA256 = {
    "model.safetensors": "891102d372688fc2a094dac56a384bc537b87c63f21f9f3dac0be2b7cbc8d86c",
    "rl_agent_config.json": "ae287b56bbcf5f8c4f4541ae9dfd00c914c4c48b940b8398c3058af37ba92bbd",
}
TRAINED = ["rapport", "paraphrasing", "active_listening", "resolved", "empathy"]    # checks this fine-tune is meant to improve
REGRESSION = ["ownership", "probing", "hostility", "escalation"]                     # checks that must not get worse

DRIVE = "/content/drive/MyDrive/laya-finetuning"
WORK = "/content/work"
os.makedirs(WORK, exist_ok=True)

# one run id per session, kept on disk so re-running this cell does not change it
_rid = f"{WORK}/run_id.txt"
if not os.path.exists(_rid) or not open(_rid).read().startswith("ft6-"):      # never reuse an older run's id (or the smoke test's)
    open(_rid, "w").write("ft6-" + datetime.datetime.now().strftime("%Y%m%d-%H%M"))
RUN_ID = open(_rid).read().strip()

print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip())
print("cuda", torch.cuda.is_available(), "| torch", torch.__version__)
print("run id:", RUN_ID)

In [ ]:
# CELL 2: mount Google Drive and create the folders.
# EXPECT: Colab asks you to allow Drive access, then "Mounted at /content/drive" and "DRIVE_OK [...]".
# If it prints DRIVE_MOUNT_FAILED, stop and paste the message.
try:
    from google.colab import drive
    drive.mount("/content/drive")
    for sub in ("checkpoints", "data", "runs"):
        os.makedirs(f"{DRIVE}/{sub}", exist_ok=True)
    print("DRIVE_OK", sorted(os.listdir(DRIVE)))
except Exception as e:
    print("DRIVE_MOUNT_FAILED:", repr(e))
    raise

In [ ]:
# CELL 3: install laya (same commit as the local v0.4.0 checkout) and the same transformers as local.
# EXPECT: no errors, then "laya 0.4.0" and "transformers 5.16.1".
# If pip says a restart is needed: restart the kernel, re-run cell 1, SKIP this cell, continue at cell 4.
!pip -q install "laya @ git+https://github.com/NandhaKishorM/laya.git@{LAYA_COMMIT}" "transformers==5.16.1"
from importlib.metadata import version
print("laya", version("laya"), "| transformers", version("transformers"), "| torch", version("torch"))

In [ ]:
# CELL 4: unpack the embedded data and the frozen production questions, and check their hashes.
# EXPECT: "data sha OK" for train_all/dev_all/test/test_long/test_calls, "frozen checks sha OK", row counts (see the table),
# and a table of rows per check. Every row must carry the same inputs hash as the frozen questions.
import base64
import gzip

FROZEN = json.loads(r"""{
  "frozen": {
    "checks": {
      "empathy": {
        "yes_means": "PASS",
        "pass_threshold": 0.78,
        "question": {
          "type": "noul",
          "instructions": "Does the agent acknowledge the customer's frustration with empathy?",
          "criteria": {
            "true": "the agent acknowledges the customer's frustration or feelings and responds with care",
            "false": "the agent ignores, brushes off or is rude about the customer's frustration"
          },
          "labels": {
            "true": "A",
            "false": "B"
          }
        }
      },
      "active_listening": {
        "yes_means": "FAIL",
        "pass_threshold": 0.19,
        "question": {
          "type": "noul",
          "instructions": "Does the second agent question ask again for information the customer already gave in their answer?",
          "criteria": {
            "true": "the second question asks again for information the customer already gave in their answer",
            "false": "the second question asks for something new, or re-checks something that may have changed"
          }
        }
      },
      "ownership": {
        "yes_means": "FAIL",
        "pass_threshold": 0.52,
        "question": {
          "type": "noul",
          "instructions": "Does the agent deflect responsibility, blame another team, or refuse to help?",
          "criteria": {
            "true": "the agent deflects, blames another team or department, refuses to help, or tells the customer to contact someone else",
            "false": "the agent takes ownership and keeps working on the customer's problem"
          },
          "labels": {
            "true": "A",
            "false": "B"
          }
        }
      },
      "rapport": {
        "yes_means": "FAIL",
        "pass_threshold": 0.5,
        "question": {
          "type": "noul",
          "instructions": "Is any agent line rude, condescending, dismissive, sarcastic or unprofessional toward the customer?",
          "criteria": {
            "true": "at least one agent line is rude, condescending, dismissive, sarcastic or unprofessional toward the customer, for example blaming the customer, talking down to them or brushing them off",
            "false": "every agent line is polite, respectful and professional"
          }
        }
      },
      "probing": {
        "yes_means": "PASS",
        "pass_threshold": 0.7,
        "question": {
          "type": "noul",
          "instructions": "Did the agent ask at least one genuine diagnostic question about the customer's problem?",
          "criteria": {
            "true": "the agent asked at least one question that investigates the cause or specifics of the problem, such as when it started, which device, what error or lights, or what was already tried",
            "false": "the agent only asked greetings, identity verification, confirmations, upsell, scheduling or closing questions"
          },
          "labels": {
            "true": "A",
            "false": "B"
          }
        }
      },
      "resolved": {
        "yes_means": "PASS",
        "pass_threshold": 0.73,
        "question": {
          "type": "noul",
          "instructions": "Did the agent explicitly confirm that the customer's issue was resolved?",
          "criteria": {
            "true": "the agent explicitly confirms the issue is resolved or fixed, usually after checking with the customer",
            "false": "the agent never confirms resolution: the call just ends, the agent assumes it is fixed, the customer says it still fails, or it is handed off"
          },
          "labels": {
            "true": "A",
            "false": "B"
          }
        }
      },
      "escalation": {
        "yes_means": "FAIL",
        "pass_threshold": 0.47,
        "question": {
          "type": "noul",
          "instructions": "Did the customer ask for a supervisor, manager or escalation that the agent refused, ignored or discouraged?",
          "criteria": {
            "true": "the customer asked for a supervisor, manager or escalation and the agent refused, ignored, discouraged or faked it",
            "false": "the customer did not ask to escalate, or the agent handled the request properly"
          }
        }
      },
      "hostility": {
        "yes_means": "FAIL",
        "pass_threshold": 0.79,
        "question": {
          "type": "noul",
          "instructions": "Is the agent hostile, abusive, insulting, threatening, mocking, or hangs up on the customer?",
          "criteria": {
            "true": "the agent insults, mocks, threatens, is abusive toward the customer, or abruptly hangs up",
            "false": "the agent stays professional and respectful, even if the customer is angry"
          }
        }
      },
      "paraphrasing": {
        "yes_means": "PASS",
        "pass_threshold": 0.5,
        "question": {
          "type": "noul",
          "instructions": "Does the agent restate or paraphrase the customer's problem back to them?",
          "criteria": {
            "true": "the agent restates the customer's problem in their own words to confirm they understood it",
            "false": "the agent never restates the customer's problem"
          }
        }
      }
    },
    "caps": {
      "EMPATHY_MAX_SNIPPETS": 2,
      "PROBING_PROBLEM_MAX_WORDS": 50,
      "PROBING_QUESTIONS_MAX_WORDS": 100,
      "RESOLVED_TAIL_LINES": 10,
      "PARAPHRASE_CUSTOMER_MAX_WORDS": 50,
      "PARAPHRASE_AGENT_MAX_WORDS": 150
    }
  },
  "sha256": "f31b14a509629d6554551702f4d6c50fd6fa41a694186b7b87d2843a78456f61",
  "inputs_sha256": "4021ce6d6740f47d3134378583b09ef98bdeaf670bcd2963f6b2555156f34475",
  "extractor_sources_sha256": {
    "laya_inputs.py": "be34e2c534c3ae3e19d8819cab9bdbc0664942a5c9798bcd18046e128d236118",
    "rule_engine.py": "5b755cea212a2dd409db3e01f64ec29ef50ab94c5e3378cdb2ea5041bc13bce9",
    "laya_windows.py": "e30484c1b03f77678a057f12aa1db3f622fddef667292bfbaf6e099f96e8015d"
  },
  "n_questions": 9,
  "qa_analysis_commit": "d03b3d47f63ce48e2a343fa9d922e8d43b85e7d7",
  "qa_analysis_branch": "laya-finetuning",
  "qa_analysis_dirty": false
}""")
DATA_B64 = {"train_all": "...", "dev_all": "...", "test": "...", "test_long": "...", "test_calls": "H4sIAAAAAAACCu1dW28buZJ+X+D8ByIvfpEEW7JlJy+DZDIXL3JDnN3BAAMYVDclcd3d1PbFijCY/36+KrJvsi5tSZZ8YgHjc6JuNu/11ccqsvj3qySVqXr1Rrx6O1JR+kb8ZowvQhNHOhqJz/JuqFXgi/9VaSLSsU4E/nunpBibqfBkJK7FWAWTn/6K3OfpWKYiMVnkJyLxZDwTMvLFzGRiqmIlYj0apyI1+Aeyz0KRuWxdiR0kOwkCERhzJ+QwVbEY67BTZJ8YyisW5k7O6IdvRmIsEzFQKhKJ9vARvo64FCo4VsMsUYkYolUd8bu8V0KLkUn5j+vKNSrr/zvaZdBg+kO1xqrF+VBzQxnNRKpDZEdFjlVZauV7+8aPs+gORYUKfUTfTSW1hbLS1GLqC3o8CKi7dSSmVJexOkFTYpNR9bNKr/rap1yVpFbN0GHUeVGWZDIQM5Uga1/OWiLQdwq9HstJIkyM/gmVTTuMTUilioGOyly/UZmeydDUgeLX9OdJ9FhHoETODmOVKEWjMDdQP/OHNLL5YAx4UCkl8k1V9NOrlnj1/xnqp02UYI79/QpVm5g45X+nswlPvAgZUUodJWmceXniV9cJd5GkyqIuEWZP5mM8PBP5KvFU5KO4FvomCXWS6Hu8SmTsSRTnUfOzaBKbocIrE6GfUjOVsW+bmCUp+ibmCnqxRv9paesUZ1wnNCBQyEkYFFupATpg15VoYXLGQn2X4STAAAUypF6sp0hlcEdPfTONaETwNqTcB3GWjF1yPBgOqUFDGSTcCHWv4tlc7ScmQHNbEIxkorx0mAU8KauVfPXPP/8gG/WdEih/btw4d3qPuWX7TPs8it5tmtyent26pLeTU+5dyONtPQU9nkgdL3jsjZV3Rw/z8vAskAMV0LMvb29u7Dzx0THxjJ7dKxq7CDjDuao4QQvoBbo5pEdjdDeXItMslsEt1Yees5DhBQ04fgKxYo87zQskBridp0vNneLpeHb+mj8zIX6cd19zRSYZqp6MZfeiT9/2+74aDl9fyO6pp07PVXdw2TvzvdPh5VDJ3uX5Zf9seKk82ev3epfDq6uB8vvD4enAG15cdb2uQs//67/+fgDJA4Ch8nmaQKrEVKdjnh9+ls4EegCyCJQhSSRJpf+/i4BXLNDqe2o/KL6Svo/BT3jYCc1KPBgRFkGOqRBz4gXGKyCMv3b/rhddhTMLNgoTpMz0xlWpqFZiTCQ+yhjo2zkCxEsGiLMfCiAuLveODz//z823zx9/+foGSldoMLRwxoyIFXc2dnxiCTWaGow6EllSARISYsyRBFzFxLEG4MiByVKn+SXoDES4Q3QDRIoyjmjG0JehnIE/DDG3Sykn+qG/85+lTBK1SlNM4JIJRcbxH0xl4mM6sjMXEjQY6GQsBnIEgkmocscFA1R+++XTt01J6hOR0ycjpZuS0d2S0N2Sz6cnnTtVY49VXxOJ9o9jmSCHRjrsvVGJVcuMwkBdEm7qwCIrVcN6DA6weBAAzgcSLXNgv1xTPcg8WZaf5k7RmMhQIgABSAlyhyIb6phHcIZx9wGhKc+XtKZLymIiUivrClukQ+Y6jxqwSo/Ukm+uSeazeSbqpNs9reiTi4u96BM3TG9WofVTLUrLwpuog/0tYqs1Q7MAH1ApZflPTs7nSr93nZM+EW0vi/tkKnp/Yz6vEhPcOwFfC4bar2AhsCHQnk7BSEoMokGpAwpYdUbzNBF5WU2w8GHmFqlsbrrMjObsUH9Xfkvw1MYHVvUzgFDHFYOXV2olLhalcQEZtf2Nm7IgCf+HLATWDEmr0hESdSJFr1OqmKtMtUCsKGb8Gh2PTIZSB8gAFbdfYBB9agjY/z85wCW1fnlbq/K7hQS/HMh1wFwk3YLeV7J4NgaAfgWQe1f7JPjiy9fP7z788vHlEP2/om+V+U8Ii/SgQgW6AMviYh2M1rz5K2qvNPC2N7fttndt1m3vWnm2m2my9nbKaAHfBZtrTHVr6I5BFTUjCp5mNNF8LUeRYfNIXpaboIuZZBO0tzOoVlyRN08SHd3Tz1HJWIlAcJcDA/VQe+j/Ib9xxbZEknlj5Iwxxkxh8IUSJGiejjXe+Opee6plp4CKY+SF/wKajRac+QXpLRlAHn3MPxLGJcrDRKR4uBmjWKkUnY5ctI93mnV6TJWU1KJWrmP4F1Jlk0QFASoMePWzgGYFigdRIMpbStQWyqGcBmtJu0u5BV8vc3gumqFfM/3smao/rUZ4csde2ZAFZPPp3H01Vo0CH0KbCicyHc82WcVjXQ5YDZQ/erB2H8Y0QVkyLdC6YhqhWJltsipfYqxKkZwnrleSiaFx4wIxdmoJyGgALxK3rNlYEeQNmUZao/ZiGK6UvAWClL29DkHylJsjSCWHZ4IgV70KgPRP9+hbusmiWEPT3ZhAxuITdFYQqEL+aJ6ThEGuIOSRp0rRJO11YqfhUGYBrcExQ4xdZ5I6jUnGQV2Uqricr5NClu1qC8WmpGUxNaEzIyIhjhuWefCixlRy+ROoQ2pADgBZRX6xnkwU2aUYM6DyhmJieIUmY8KtjnhvohPKnxQ306iEAcznx860xXyVqwHGU656bf1qdWbhwN9UokgpQh1lqcpbgkdsnjNRkcdvbp1OaA0eybZrOZI66izznTFYHZ1kP6CTbDXIdQsf2XCJE727GOe6a31kv769/jCPczSh4zmIGwRZlO6WJZ32DuggAz2ayAjiBTBQJO4YKz/zWAZz4QMd9qzCxoyxFu8ChQB1CRaKnN4BnluwQqQ7YBljkI0CXYhB5bkCdFRHfMtx8qREQA/CEXNSn0GhqFJHfDKFr2shQi9A5sch8nZIvD8E3gZ5myPuUqQ9+nN2589ZszGg28Ch0wj2ljp0Doh93csDOnOuU2sg/lV0Ra+zKy5Wyb8EwkNQtLIiy1F4N0SuLGkTUN+ECFa9Metp4dHX8sJ8Ld01vpZmNLGpr2VPWHl29Qz8LP+BfHEzl8kj1/Ht7dTG0YNw9CBs5UHorvYgNCOIDT0I+8G7q6u9Ow/ysyq0ABiLmxSTJ8Tk+ChjzwhyYpZc5fMduSGtHf7E7T1Jp8AitJ/M0NY6EpoIenmQuVlG0pFkA4CGntDE6MyfsvmZN2WOlJVIFUKJ5pAjPQ9MsJIWw4bu74ACFo5LqoCvh7zyTcmgjZWN+3wAoKE5OcGkr5kd6SU5fil1Cc0QHcDLfQF46I7FOZCMMxWNaRcN8Rn0JXAcsOZJqIGgKCALbftRXSWBh2Za9qbF9k+QakpzNOy9uN3vvbXHY3qLEay30e73hEXbYV0FxZJARqPZTmHs9evDbn8HYXPQ4Rh6KAaKxt4iFSbVlMROOqyiUb22aJXoUYT32QQ/yY5GYtthguXLSBeC7AP5mHAwWtEvz4Qd8Se5yTB5rG4PT0gqS6DRvFSwGNMiVyM0LtEtglHJehgTlvyY38j5SY5IZ/Vbgc27xeQmWLwNBm+Ovdth7lqsPZr29rVVu9fAstcI9h61VXuP2Hd2fnZA8x7j1eOoVcVk1gzmdsPGymJXouYuaFulhQ0heFdUryy5xPSG7O9ov/vh7Hfr+eBKA14zNrjIgLfQ2bFHSKwdXzngbukfhRduZt0rNz830gvtzXC9vTlSHw2CR4PgVgbB3mqDYDNe2dQguEf8POvv3Sz4LYsj0Rdvy/n9ZjueB8hLaHewv5bwcdFXrujAHqqwFdghstj0t4FOUhU9etWXKDJqOdkpYYQwh3y3bKbSEf7XCkmddOTyOOItyfnizXbParRxxVYLTLYt8QEgLCxkWDtwEqkpQ0yMyUpyk1TeMd6FcsZOLVKsERTrIqFeMAJrCdKDbzaX80VZPReBP6+6AS5O92s/+zgTofId2LMCoRNOqbUphKo8keR8j7QJn5yXMo71fe6RvCcLNISaLE6QcgmVVonHUBw2yiLwDTXh9+JGfwfpQvL/ziL001AP01mbzpuVx2ZzUhP+VOTl0wRDD1fsZV8wEKH0ZtCRgSb7V0fcsHmGzw/Ylcrn2IdgRFk4UHFHvLcJNcQb6ziJxR5x/wCtGcysKOFLrehYryQ+Rycd+IwVT3Ky5bDBJcWksvblyHQefMenFYgykp3GluxONHBdit17BZvk1ZRHb/3jdrRDbkc7b2C0Ol8MOuebb0ebuGl8m0/jOeBB6wxo7G4t94fcmzZ5D7V/Jc5EV5xWjFhuUrCoFAstnrlDaPk4HdcMLctRZHGO7mBjLY+e+Ej2gA+y+tWuMaJa4EN8K4ptiCT1w1MLoNHmtgXeHC1SL84idb7GItUI8xpbpA6Bd2enr/dumnoiPHkosUAH9N5YTzahB2hzgDnjDgwmGuspnc5a7HWng+2GTrhz3dxSgE5bUp04vFYDuXUFJC7LpJYnZekrqMs0RNpWcZjT5c9FpioI6jq/2iW0IqFFJm2eXSLGqbxTjKG2l5i53ik1SYiRMCDMLaSqnGJj8auOympzxvltmXZz+avl8ewEsFv1lp1f7pdwHGqpU+6oqq5Cqir8yanQ8YzzcznjvFYHrzzk3AgCFh1yfi4AUIuXsNfjznz0Akwu4m1pRcQBtnVRNUkB3KRyNmDHSasQ6S8qUoGZqA4HkSFT1zWFG4Ocgd5pOpLiVbcbkr0036GTRwiTeCjb91pNBTW9xVtzWKqrtnVLBj02wCNplQNwaAdN7qU45knVscZRWTwqa/CRa2hZA9lnjT1lU/D/WhBw2wct8TXDuKcd8Yd1NvDymjgKG4DRKcih/O4PmZtFJddUkNJGA4iqWg06VYMEsm6dGdaIHHC9begf9EGZ22dobdvKfI+81eTTsRET6hH0pQsrZMPoRIYhe8ArGeIM2svofGQpvlQp28kYjjHRczJis4eEYoLiS99U13q/mnhEwB5ikK85CaWfGzWm8LEbXNfBoSG9UItb8ZbMpfVxLcNWIAX6uAjXE0Iay+WIdR2lbOZOis49RlJ/cWfAL9aeAb9YrAYuNjoDPjag1rfeWOpoTgFgdsfB7NY2t4ka4F5rthK77B4qpvofNOtiDFqrCDKTyzcHNXSC5yC2FihxmMX0ZBWEHsX1xe3svli7s3tjeV3kmTqIvHYPGeL8dyz9TYt4V86nWGCZ78uShgxy9r6cVNlNPicLGdWfFPWQD98ZEvP8JFtCnBFLLXpPW3V44WgdGrYaqQY3IeKAAbbTHDMsEDFvHfw8bnGU1g4dxfYNZcHRFT+Z0oF1MM64nis+liPuhhsekBPuggvungPuQmc10lVHL+O+dsZfNHAyNlIbj9oZfxjdcXZxQDfjath+FsvQajyNBUpkt+vUsrBCNz3d4rXWtqoCLI0Ju2Dj1VgZnceR86Ob84UFzrhY4+VsxtWbBs44zOL6vPsMduD/qKx9w/jkq4zH7SZW2/Ya62x7V4bZ9uNRvr0Cno/7/I/7/Lfa53+xep9/M5bccJ//YfC6d7BAID/TPPigB7GMZ51Op/B6fVXKr4cCobfORBErtDhyoEFQJQISkKlSd851TjGQWNfne/ap9/xMUSa5FWIuMMgfNhoRM8DAVshu+7f+8mpCEhQulkG9ksf7vH6h4v22dL9DrcI6bQlH0XwFLOFt9PZnbCbI9PuKclp2832Z4itHY+tUybjTNAl+c3lRuasAyejIfh7KiVVObaMieSiZBUeG30DhvK0GhUNJR2/QyzMv99eal/uLEbC/kXnZyd6DYw9KTWY7hr2z80P5gN7NFJ/ZPorTSxansx9KnPqH9NBoWugValbOswJa5BXrf7t6gJZ0pIAWJ+VZ7cqSboS+4f06NMlb4q9XNyEZIb4EJk3+egV9WlfdtCkvAbntzKd0lyldY+Jrv1PahTrW9VI9CW2NTlzBUgtXnTUrCdNTEqW1BGkBMdoJIVpOhHZEgBoRnxWQfXRa7Mtp0W/gtGiEno9yWuyRkfR7B/RUNIC8Ha90Ksb5JVDaZDVU5rISd3e+bJoLt/MIRH/UkmuZe2EFhzw6FV6YU6G/xqnQjFI2dSrsERC7z8KT8OOzy82cCuttZu2F5rL2jvRHe7laOFr9j1b/raz+/dVW/2Y0s6HVf5+A2nsWwX0WocKC2D2NKCnnf3a6KITPI+DhGKLnGYboWSeizSL0NJLVlRF6Fp1b2J/Q9i73HqAnv0uJbiWnfRTi18BM7U27XwIodxrYjzLjaIHsdg9AdwD7fGJaxvaScFLopNg9DgDIJ7WL1YsZU9is6tW+ckq2VnswLj9tmh+QI+rB956ksbRX7k4xofjoK8UVzDfLpaZ+GMpF8LJXpVt65hlnPeZhYCuY58JhcIRCk9+qMvEM38Bey4r2F4hUTvJdJPYMvbWQ8Id3dBMJ3xAcVT5mhLJbnwPkOqVL9QBHQ9quR6hE1xPnTR4Z1yuogq1+FkJ/Tya2d5lrDQFyZc34a0XoEwnwqRiPvZmYBDbmj+Uj6ESoXnu/VMK9V9zaV41LUlb5HW+a6RSdkzPJ8rJ2Jhi2EOQSybBytG/k7pniu6BpFx9WRFxmbXV79Ca+PPfH5Vpv4uVitL7cyP0xcWA1f7RYhoPAPd5ddI+Lg50po9gSfPS8Jb5ZXn4Uq5cqVmc/mFgd0q/4ecwRNyg2siURc/SBp4vSvE5lysPmYWhnWsim6OykuDKN1ixkEqrRBBvz2TeYKRy2g9fZmGg+h+HKQwRcl246NvX4dJklX76WX4d27a6pdJkQT7FmyJlyvOWkTskphT1wZDMP2PtnY0ezV5AjkrhmFs7HndPBJ6WBO6B/G9G+J6F7mxGypkRsrQI5+jj3Fv7xsoGTsxGYPy784z6J0tXFQS8tWQqJT7xgK+uwAHqfaj1XFtrtindKeWPxMx1VsKjS23bBV79YZZHqeNSKsO7sLHVg2T1rqe7R+fnCnJ+Xa5yfzZhvU+fnXoHyWbg/jyTYItlmPtJd2CDbG5of20t0QvtxKH/0pR59qVv5Ui9X+1KbsdmGvtS94nOvu3dvavX8/MFweU9unGqQcacHdurbqQYH2FS7HIObPp/gputQaGVs00YotCi26aFR6HXvUDFNsZCMPB2wLY8s3S3xDlMwKuHhxq1/0RUJ+/9Jgjich2cCd/KbgIEueKKtdEnH7s4tPYlfQAk8s2AnsCYVhycKfUfq1BqXYuXNvKAWLpT4B0HAJHO3pdBEHCgU647pV45csr60ccEnypBjIJQRJjjvHAEIhHI2UO7YREEk6S5kZpK8NotNfTF/T8bQ0Yj3DZKvAv9wbeerkEEpZycWuxR+ZHFCe46rm3yPh8ReXATOq7UROK8Wg9XVRhE4PSvGc1g1CLIo3e1GltfnB/TlfJw1hyEeTUsWiHXTkDPN50liRBICKNxW3nemCPNhZ0qqKT7SN7vwoihNQwxT58E168uwc3PMLLGyKUY2wMbdYeKmWLgEA48eiX2durpq4JBohEePOnW1N1A6Ozvkoat3Nxei/+HbBvSmkodJK9ewN+E7c+HpKsD1FFyodlCqhorb86Tld54voU5H98BLu1fqao1/oBmXanqv1P7IVP8Z+AaeC6nazDb/tbijvDHqthvg69FefrSXb2Uvv1ptL29Gthray/eGV5eX+7GV/xuBUVHNWcQAAA=="}
DATA_SHA256 = {"train_all": "9269343a4ec667091be723f1088f481a5929e704dbcf229f63e9d4cad5645964", "dev_all": "ac80492505e85d2a00ba41b8669dedd2d7976c274de1aa0681889fcaca7d286b", "test": "f379d4c5ea17e656b1fe8792121013f84c4b0cad917936b46476a4622dcd6e37", "test_long": "1b925ff096259c608de0b355304c172112ac0b293522ede9961bc8ffa7ae5b51", "test_calls": "1585163ef80a7b0687282f3d3ffb869ba7f6fd057d1590396d8f5c0c92f9d610"}


def canonical_sha(obj):
    text = json.dumps(obj, sort_keys=True, ensure_ascii=False, separators=(",", ":"))
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


assert canonical_sha(FROZEN["frozen"]) == FROZEN["sha256"], "frozen checks were altered"
CHECKS = FROZEN["frozen"]["checks"]
print("frozen checks sha OK:", FROZEN["sha256"][:12] + "...", "|", len(CHECKS), "questions")

DATA_DIR = f"{WORK}/data"
os.makedirs(DATA_DIR, exist_ok=True)
SPLIT = {}
for name, b64 in DATA_B64.items():
    raw = gzip.decompress(base64.b64decode(b64))
    assert hashlib.sha256(raw).hexdigest() == DATA_SHA256[name], f"{name}: embedded data was altered"
    open(f"{DATA_DIR}/{name}.jsonl", "wb").write(raw)
    SPLIT[name] = [json.loads(l) for l in raw.decode("utf-8").splitlines() if l.strip()]
    print(f"data sha OK: {name:<10} {len(SPLIT[name])} rows")
ACCEPTED = set(["4021ce6d6740f47d3134378583b09ef98bdeaf670bcd2963f6b2555156f34475", "66deff95a20ce04e2b731dc0f7fea374761f7eca36337f88bed6ff0bcf582c2e"]) | {FROZEN["inputs_sha256"]}     # rows record the extractor hash that built them
for s in SPLIT.values():
    assert all(r["meta"]["inputs_sha256"] in ACCEPTED for r in s), "rows were built from inputs that are not in the accepted list"
print("row input hashes (accepted):", sorted({r["meta"]["inputs_sha256"][:12] for s in SPLIT.values() for r in s}))

import collections
tbl = collections.defaultdict(lambda: collections.Counter())
for name, rows in SPLIT.items():
    for r in rows:
        tbl[r["meta"]["check"]][name] += 1
print(f"{'check':<18}" + "".join(f"{n:>12}" for n in SPLIT))
for c, cnt in tbl.items():
    print(f"{c:<18}" + "".join(f"{cnt[n]:>12}" for n in SPLIT))

In [ ]:
# CELL 5: download the English checkpoint at the pinned revision and compare it with the local copy.
# EXPECT: files listed (model.safetensors about 842 MB), "weights identical to local copy: True",
# "config identical to local copy: True", and max_len 512 / head_max_len 192.
from huggingface_hub import snapshot_download


def sha256_file(path, chunk=1 << 24):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while block := f.read(chunk):
            h.update(block)
    return h.hexdigest()


BASE = snapshot_download(BASE_REPO, revision=BASE_REV, local_dir=f"{WORK}/base-english",
                         allow_patterns=["model.safetensors", "rl_agent_config.json", "encoder/*", "tokenizer/*"])
for dp, _, fns in sorted(os.walk(BASE)):
    for fn in sorted(fns):
        p = os.path.join(dp, fn)
        if "/.cache" not in p.replace("\\", "/"):
            print(f"{os.path.getsize(p) / 1e6:9.1f} MB  {os.path.relpath(p, BASE)}")

BASE_SHA256 = {name: sha256_file(f"{BASE}/{name}") for name in LOCAL_BASE_SHA256}
same_w = BASE_SHA256["model.safetensors"] == LOCAL_BASE_SHA256["model.safetensors"]
same_c = BASE_SHA256["rl_agent_config.json"] == LOCAL_BASE_SHA256["rl_agent_config.json"]
print("weights identical to local copy:", same_w)
print("config identical to local copy:", same_c)
BASE_CFG = json.load(open(f"{BASE}/rl_agent_config.json"))
print("max_len", BASE_CFG["max_len"], "| head_max_len", BASE_CFG["head_max_len"], "| encoder", BASE_CFG["encoder"])
assert same_w and same_c, "downloaded base differs from the local snapshot: stop and tell Claude"

In [ ]:
# CELL 6: helpers for safe copying to Drive and for running laya-train with live output (also writes the GPU-fix launcher).
# Also writes laya_train_cuda.py: laya 0.4.0's trainer crashes on a GPU ("tensors on the same device ...") because it
# evaluates before moving the model to the GPU. This wrapper does that move first; the package itself is not edited.
# EXPECT: no output other than "helpers ready".
LAUNCHER = f"{WORK}/laya_train_cuda.py"
with open(LAUNCHER, "w", encoding="utf-8", newline="\n") as f:
    f.write(r'''"""laya-train with one fix for GPUs. Same arguments as `laya-train` / `python -m laya.train_cli`.

Why it exists: in laya 0.4.0, finetune() evaluates the calibration slice (or the --eval file) BEFORE training,
but load_checkpoint() returns the model on the CPU and only train_model() moves it to the GPU. On CUDA that
evaluation crashes with "Expected all tensors to be on the same device ... index_select". Laya's own tests run
on CPU, so they never hit it. This wrapper moves the model to the target device before every
calibration_records() call. Nothing in the installed laya package is edited.

Use it exactly like laya-train:
    python laya_train_cuda.py --data train.jsonl --base <dir> --out <dir> --loss soft-ce --epochs 4 ...
Drop this wrapper if a later laya release fixes it upstream (run once without it to find out).
"""
import sys

import laya.train as _train

_original = _train.calibration_records


def _calibration_records_on_device(model, tok, items, device, *args, **kwargs):
    model.to(device)      # a no-op when it is already there (after training)
    return _original(model, tok, items, device, *args, **kwargs)


# finetune() looks calibration_records up in laya.train's globals at call time, so this takes effect
_train.calibration_records = _calibration_records_on_device

from laya.train_cli import main  # noqa: E402  (imported after the patch on purpose)

if __name__ == "__main__":
    sys.exit(main())
''')

EPOCH_RE = re.compile(r"epoch (\d+)/(\d+) mean loss")


class SourceChanged(Exception):
    pass


class CopyMismatch(Exception):
    pass


def tree_stats(path, ignore=()):
    """Sorted (relative path, size, mtime_ns) for every file under path. Files that vanish mid-scan are skipped."""
    out = []
    for dp, dns, fns in os.walk(path):
        if dp == path:
            dns[:] = [d for d in dns if d not in ignore]
        for fn in fns:
            p = os.path.join(dp, fn)
            try:
                st = os.stat(p)
            except FileNotFoundError:
                continue
            out.append((os.path.relpath(p, path), st.st_size, st.st_mtime_ns))
    return sorted(out)


def wait_stable(path, quiet_s=30, poll_s=3, timeout_s=1800, newer_than=None, ignore=()):
    """Return once no file under path changed size or mtime for quiet_s seconds.

    With newer_than (a time.time() value), also require that something was written at or after it, so an
    older, already-stable copy is never mistaken for the checkpoint that is about to be written.
    """
    t0, last, since = time.time(), None, time.time()
    while time.time() - t0 < timeout_s:
        snap = tree_stats(path, ignore)
        if snap != last:
            last, since = snap, time.time()
        fresh = newer_than is None or (snap and max(m for _, _, m in snap) / 1e9 >= newer_than)
        if snap and fresh and time.time() - since >= quiet_s:
            return snap
        time.sleep(poll_s)
    raise TimeoutError(f"{path} was not stable within {timeout_s}s")


def copy_verified(src, dst, tag, ignore=()):
    """Copy src to a temp folder next to dst, verify it, then rename it into place.

    Raises SourceChanged if src changed while copying, CopyMismatch if the copy differs. dst is only
    replaced after both checks pass, so a half-written copy never sits under the final name.
    Returns (number of files, total bytes).
    """
    tmp = f"{dst}.tmp-{tag}"
    if os.path.exists(tmp):
        shutil.rmtree(tmp)
    before = tree_stats(src, ignore)
    shutil.copytree(src, tmp, ignore=shutil.ignore_patterns(*ignore) if ignore else None)
    names_sizes = lambda stats: [(r, z) for r, z, _ in stats]
    if names_sizes(tree_stats(tmp)) != names_sizes(before):
        shutil.rmtree(tmp, ignore_errors=True)
        raise CopyMismatch(f"{tmp} differs from {src}")
    if tree_stats(src, ignore) != before:
        shutil.rmtree(tmp, ignore_errors=True)
        raise SourceChanged(src)
    old = None
    if os.path.exists(dst):
        old = f"{dst}.old-{tag}"
        os.rename(dst, old)
    os.rename(tmp, dst)
    if old:
        shutil.rmtree(old, ignore_errors=True)
    return len(before), sum(z for _, z, _ in before)


class CkptWatcher:
    """Copies src (checkpoint_latest/) to Drive each time an epoch finishes AND the folder has stopped changing.

    laya-train prints the "epoch N/M mean loss" line BEFORE it writes the checkpoint, so the line alone is
    not enough: after it, wait until nothing was written for quiet_s seconds (and something was written
    after the line), copy to a temp folder on Drive, verify, rename. finish() does one last pass after the
    process exits, because laya-train rewrites the config after calibration.
    Messages are queued and printed by the notebook's main thread (flush), never from this thread.
    """

    def __init__(self, src, dst, quiet_s=30):
        self.src, self.dst, self.quiet_s = src, dst, quiet_s
        self.work, self.msgs = queue.Queue(), queue.Queue()
        self.copies, self.errors = [], []
        self.thread = threading.Thread(target=self._loop, daemon=True)
        self.thread.start()

    def say(self, text):
        self.msgs.put(f"[watcher {time.strftime('%H:%M:%S')}] {text}")

    def feed(self, item):
        t, line = item
        m = EPOCH_RE.search(line)
        if m:
            self.work.put(("epoch", int(m.group(1)), t))

    def finish(self):
        self.work.put(("final", 0, time.time()))

    def stop(self):
        """Training failed: end the thread without waiting for a checkpoint that may never come."""
        self.work.put(("stop", 0, time.time()))

    def flush(self):
        while not self.msgs.empty():
            print(self.msgs.get())

    def _loop(self):
        while True:
            kind, n, t = self.work.get()
            if kind == "stop":
                self.say("training failed, nothing more to copy")
                return
            if kind == "final" and not os.path.isdir(self.src):
                self.say("no checkpoint_latest was written, nothing to copy")
                return
            tag = f"epoch{n}" if kind == "epoch" else "final"
            quiet = self.quiet_s if kind == "epoch" else 10
            try:
                for attempt in range(1, 4):
                    wait_stable(self.src, quiet_s=quiet, newer_than=(t - 2) if kind == "epoch" else None)
                    try:
                        files, size = copy_verified(self.src, self.dst, tag)
                    except SourceChanged:
                        self.say(f"{tag}: folder changed while copying, retry {attempt}/3")
                        continue
                    self.say(f"{tag}: quiet for {quiet}s -> copied to tmp -> verified {files} files / "
                             f"{size / 1e6:.0f} MB -> renamed to {os.path.basename(self.dst)}")
                    self.copies.append((tag, files, size))
                    break
                else:
                    raise RuntimeError(f"{tag}: source kept changing")
            except Exception as e:  # reported by the notebook; never kills training
                self.errors.append(f"{tag}: {e!r}")
                self.say(f"ERROR {tag}: {e!r}")
            if kind == "final":
                return


def run_logged(cmd, log_path, on_item=None):
    """Run cmd, echo its output live, save it to log_path. on_item((time, line)) sees every line; on_item(None) is a tick.

    Returns (return code, [(time, line), ...]).
    """
    q, events = queue.Queue(), []
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)

    def reader():
        for line in proc.stdout:
            q.put((time.time(), line.rstrip("\n")))
        q.put(None)

    threading.Thread(target=reader, daemon=True).start()
    with open(log_path, "w", encoding="utf-8") as log:
        while True:
            try:
                item = q.get(timeout=1)
            except queue.Empty:
                if on_item:
                    on_item(None)
                continue
            if item is None:
                break
            events.append(item)
            log.write(item[1] + "\n")
            log.flush()
            print(item[1])
            if on_item:
                on_item(item)
    return proc.wait(), events


print("helpers ready")

In [ ]:
# CELL 7: scoring and metric helpers (balanced accuracy, AUC, bootstrap intervals by pair, ECE, replay drift).
# EXPECT: only "eval helpers ready".
import random

import laya
import numpy as np


def predict_rows(agent, rows):
    """{"<row id>/<question id>": P(yes)} for every question of every row."""
    out = {}
    for r in rows:
        res = agent.predict(r["state"], r["questions"])
        for qid in r["questions"]:
            out[f"{r['meta']['id']}/{qid}"] = float(res["answers"][qid]["noul"])
    return out


def p_pass(check, p_yes):
    return p_yes if CHECKS[check]["yes_means"] == "PASS" else 1.0 - p_yes


def supervised(rows):
    return [r for r in rows if r["meta"]["check"] != "replay"]


def by_check(rows, preds):
    """{check: [(P(pass), is_pass, pair_id), ...]}"""
    d = {}
    for r in supervised(rows):
        c = r["meta"]["check"]
        d.setdefault(c, []).append((p_pass(c, preds[f"{r['meta']['id']}/{c}"]), r["meta"]["label"] == "PASS", r["meta"]["pair_id"]))
    return d


def bal_acc(items, thr):
    pos = [p for p, y, _ in items if y]
    neg = [p for p, y, _ in items if not y]
    if not pos or not neg:
        return float("nan")
    return 0.5 * (sum(p >= thr for p in pos) / len(pos) + sum(p < thr for p in neg) / len(neg))


def auc(items):
    pos = [p for p, y, _ in items if y]
    neg = [p for p, y, _ in items if not y]
    if not pos or not neg:
        return float("nan")
    wins = sum((a > b) + 0.5 * (a == b) for a in pos for b in neg)
    return wins / (len(pos) * len(neg))


def best_cut(items):
    """Cutoff with the best balanced accuracy on these items; ties go to the one nearest 0.5."""
    ps = sorted({p for p, _, _ in items})
    cands = [0.0] + [(a + b) / 2 for a, b in zip(ps, ps[1:])] + [1.0]
    scored = [(bal_acc(items, t), -abs(t - 0.5), t) for t in cands]
    best = max(scored)
    return best[2], best[0]


def boot_ci(items, thr, n=1000, seed=0):
    """95% interval of balanced accuracy, resampling PAIRS (a pair's two rows always travel together)."""
    groups = {}
    for it in items:
        groups.setdefault(it[2], []).append(it)
    keys = list(groups)
    rng = random.Random(seed)
    vals = []
    for _ in range(n):
        sample = [it for k in (rng.choice(keys) for _ in keys) for it in groups[k]]
        v = bal_acc(sample, thr)
        if v == v:
            vals.append(v)
    vals.sort()
    return vals[int(0.025 * len(vals))], vals[int(0.975 * len(vals)) - 1]


def ece(rows, preds, bins=10):
    """Expected calibration error of P(yes) against the yes/no truth, pooled over all supervised rows."""
    pts = [(preds[f"{r['meta']['id']}/{r['meta']['check']}"], bool(r["expected"][r["meta"]["check"]])) for r in supervised(rows)]
    total, err = len(pts), 0.0
    for b in range(bins):
        lo, hi = b / bins, (b + 1) / bins
        sel = [(p, y) for p, y in pts if lo <= p < hi or (b == bins - 1 and p == 1.0)]
        if sel:
            err += len(sel) / total * abs(sum(p for p, _ in sel) / len(sel) - sum(y for _, y in sel) / len(sel))
    return err


def replay_drift(rows, preds):
    """How far the tuned model's answers to generic custom-criteria questions moved from the base model's own."""
    diffs, agree, n = [], 0, 0
    for r in rows:
        if r["meta"]["check"] != "replay":
            continue
        for qid, g in r["gold"].items():
            base_p = g["probabilities"]["true"]
            p = preds[f"{r['meta']['id']}/{qid}"]
            diffs.append(abs(p - base_p))
            agree += (p > 0.5) == (base_p > 0.5)
            n += 1
    return {"items": n, "mean_abs_diff": float(np.mean(diffs)), "verdict_agreement": agree / n}


def evaluate(dev_rows, dev_preds, test_rows, test_preds, cuts=None):
    """Per-check metrics on test. Cutoffs for the TRAINED checks are tuned on dev (or passed in); regression checks keep the frozen cutoff."""
    dv, ts = by_check(dev_rows, dev_preds), by_check(test_rows, test_preds)
    out = {"checks": {}, "ece_test": ece(test_rows, test_preds), "replay_dev": replay_drift(dev_rows, dev_preds)}
    for c in TRAINED + REGRESSION:
        frozen_cut = CHECKS[c]["pass_threshold"]
        row = {"n_test": len(ts[c]), "frozen_cut": frozen_cut, "auc_test": auc(ts[c]), "auc_dev": auc(dv[c]),
               "ba_test_frozen_cut": bal_acc(ts[c], frozen_cut), "ba_dev_frozen_cut": bal_acc(dv[c], frozen_cut)}
        if c in TRAINED:
            cut = cuts[c] if cuts else best_cut(dv[c])[0]
            row.update({"tuned_cut": cut, "ba_dev_tuned": bal_acc(dv[c], cut), "ba_test_tuned": bal_acc(ts[c], cut)})
            row["ba_test_tuned_ci"] = boot_ci(ts[c], cut)
        else:
            row["ba_test_frozen_ci"] = boot_ci(ts[c], frozen_cut)
        out["checks"][c] = row
    out["mean_auc_dev_trained"] = float(np.mean([out["checks"][c]["auc_dev"] for c in TRAINED]))
    out["mean_ba_test_trained"] = float(np.mean([out["checks"][c]["ba_test_tuned"] for c in TRAINED]))
    return out


def evaluate_long(rows, preds, res):
    """Long-snippet test (Rapport, Paraphrasing, Resolved padded with neutral agent lines). Uses the dev-tuned cutoff of `res` when given."""
    ts, out = by_check(rows, preds), {}
    for c in ts:
        if not any(y for _, y, _ in ts[c]) or all(y for _, y, _ in ts[c]):      # needs both PASS and FAIL rows to score
            continue
        cut = res["checks"][c].get("tuned_cut", CHECKS[c]["pass_threshold"]) if res else CHECKS[c]["pass_threshold"]
        out[c] = {"n": len(ts[c]), "cut": cut, "ba": bal_acc(ts[c], cut), "ba_frozen_cut": bal_acc(ts[c], CHECKS[c]["pass_threshold"]),
                  "auc": auc(ts[c]), "ci": boot_ci(ts[c], cut)}
    return out


DEV, TEST, TEST_LONG, TEST_CALLS = SPLIT["dev_all"], SPLIT["test"], SPLIT["test_long"], SPLIT["test_calls"]
print("eval helpers ready")

In [ ]:
# CELL 8: baseline. The untouched base model answers every dev and test question (about 1 minute).
# EXPECT: a table of the BASE model's test scores per check: balanced accuracy at the production cutoff and AUC.
# AUC 0.50 = no signal; the base model is expected to be weak on rapport. Saved to /content/work/preds_base.json.
t0 = time.time()
base_agent = laya.load(BASE)
PRED = {"base": {"dev": predict_rows(base_agent, DEV), "test": predict_rows(base_agent, TEST), "test_long": predict_rows(base_agent, TEST_LONG),
                 "test_calls": predict_rows(base_agent, TEST_CALLS)}}
del base_agent
torch.cuda.empty_cache()
json.dump(PRED["base"], open(f"{WORK}/preds_base.json", "w"))
RES = {"base": evaluate(DEV, PRED["base"]["dev"], TEST, PRED["base"]["test"])}
RES["base"]["long"] = evaluate_long(TEST_LONG, PRED["base"]["test_long"], None)
RES["base"]["nat"] = evaluate_long(TEST_CALLS, PRED["base"]["test_calls"], None)
print(f"base scored in {time.time() - t0:.0f}s")
print(f"{'check':<18}{'n':>5}{'BA@prod cut':>13}{'AUC':>7}")
for c in TRAINED + REGRESSION:
    r = RES["base"]["checks"][c]
    print(f"{c:<18}{r['n_test']:>5}{r['ba_test_frozen_cut']:>13.3f}{r['auc_test']:>7.3f}")
print("ECE (test):", round(RES["base"]["ece_test"], 3))
print("natural-call snippets (cut by the extractors from complete calls, 8 unseen-industry calls):", {c: f"BA {v['ba']:.2f} AUC {v['auc']:.2f}" for c, v in RES["base"]["nat"].items()})
print("long snippets (padded, closer to real call length):", {c: f"BA {v['ba']:.2f} AUC {v['auc']:.2f}" for c, v in RES["base"]["long"].items()})

In [ ]:
# CELL 9: train two variants on train_all (soft-ce and rlcd), 4 epochs each, and score each on dev and test.
# laya-train writes to /content (local disk); a watcher copies checkpoint_latest/ to Drive once it is fully written.
# EXPECT per variant (about 7-10 min, ~1,270 rows x 4 epochs at roughly 14 rows/s): "train items ..., calibration items ...",
# "epoch N/4 mean loss ..." with the loss falling, "Training complete!", watcher lines, then "exit code 0".
# A RuntimeWarning about calibration is fine. If one variant fails the other still runs; paste the error.
AG_OVERSAMPLE = 2     # 0: the original two loss variants. N >= 2: soft-ce on the rows as they are, and soft-ce with the Antigravity rows repeated N times
SOFT = ["--loss", "soft-ce", "--label-smoothing", "0.05"]
if AG_OVERSAMPLE:
    RUNS = [("softce", SOFT), (f"softce_ag{AG_OVERSAMPLE}", SOFT)]
    _rows = [l.rstrip("\n") + "\n" for l in open(f"{DATA_DIR}/train_all.jsonl", encoding="utf-8") if l.strip()]
    _ag = [l for l in _rows if json.loads(l)["meta"].get("source") == "antigravity-agt"]
    with open(f"{DATA_DIR}/train_all_ag{AG_OVERSAMPLE}.jsonl", "w", encoding="utf-8", newline="\n") as _fh:
        _fh.writelines(_rows + _ag * (AG_OVERSAMPLE - 1))
    DATA_FOR = {"softce": f"{DATA_DIR}/train_all.jsonl", f"softce_ag{AG_OVERSAMPLE}": f"{DATA_DIR}/train_all_ag{AG_OVERSAMPLE}.jsonl"}
    print(f"{len(_ag)} Antigravity rows; the second variant sees {len(_rows) + len(_ag) * (AG_OVERSAMPLE - 1)} rows ({AG_OVERSAMPLE}x Antigravity)")
else:
    RUNS = [("softce", SOFT), ("rlcd", ["--loss", "rlcd"])]
    DATA_FOR = {n: f"{DATA_DIR}/train_all.jsonl" for n, _ in RUNS}
COMMON = ["--epochs", "4", "--micro-batch", "8", "--grad-accum", "2", "--seed", "0", "--max-len", "512", "--head-max-len", "192"]
DRIVE_RUN = f"{DRIVE}/checkpoints/{RUN_ID}"
os.makedirs(DRIVE_RUN, exist_ok=True)

# dry run first: prints how many items laya-train builds and skips (should skip none)
rc, ev = run_logged([sys.executable, "-u", LAUNCHER, "--data", f"{DATA_DIR}/train_all.jsonl", "--base", BASE, "--dry-run",
                     "--max-len", "512", "--head-max-len", "192"], f"{WORK}/dry_run.log")
assert rc == 0, "dry run failed: paste the output above"

TRAIN_INFO = {}
for name, loss_args in RUNS:
    OUT = f"{WORK}/ckpt-{name}"
    shutil.rmtree(OUT, ignore_errors=True)
    cmd = [sys.executable, "-u", LAUNCHER, "--data", DATA_FOR[name], "--base", BASE, "--out", OUT] + loss_args + COMMON
    print(f"\n===== {name}: {' '.join(cmd[3:])}")
    watcher = CkptWatcher(f"{OUT}/checkpoint_latest", f"{DRIVE_RUN}/checkpoint_latest-{name}", quiet_s=30)
    t0 = time.time()
    rc, events = run_logged(cmd, f"{WORK}/train_{name}.log", on_item=lambda it: ((watcher.feed(it) if it else None), watcher.flush()))
    (watcher.finish if rc == 0 else watcher.stop)()
    watcher.thread.join(timeout=300)
    watcher.flush()
    wall = time.time() - t0
    print(f"{name}: exit code {rc} | wall {wall:.0f}s | watcher copies {len(watcher.copies)} | watcher errors {watcher.errors}")
    TRAIN_INFO[name] = {"rc": rc, "wall_s": round(wall), "cmd": " ".join(cmd), "watcher_errors": watcher.errors,
                        "losses": [l for _, l in events if "mean loss" in l]}
    if rc != 0:
        print(f"{name} FAILED: skipping it, the error is in the output above")
        continue
    t1 = time.time()
    agent = laya.load(OUT)
    PRED[name] = {"dev": predict_rows(agent, DEV), "test": predict_rows(agent, TEST), "test_long": predict_rows(agent, TEST_LONG),
                 "test_calls": predict_rows(agent, TEST_CALLS)}
    del agent
    torch.cuda.empty_cache()
    json.dump(PRED[name], open(f"{WORK}/preds_{name}.json", "w"))
    RES[name] = evaluate(DEV, PRED[name]["dev"], TEST, PRED[name]["test"])
    RES[name]["long"] = evaluate_long(TEST_LONG, PRED[name]["test_long"], RES[name])
    RES[name]["nat"] = evaluate_long(TEST_CALLS, PRED[name]["test_calls"], RES[name])
    print(f"{name}: scored in {time.time() - t1:.0f}s | mean dev AUC (trained checks) {RES[name]['mean_auc_dev_trained']:.3f}")
assert any(n in RES for n, _ in RUNS), "no variant trained successfully: paste the errors above"

In [ ]:
# CELL 10: compare base vs tuned, pick the better variant on DEV, print the tables.
# EXPECT: WINNER line, then one table per variant with test balanced accuracy (BA) with a 95% interval (resampled by pair),
# AUC, tuned cutoff (from dev) and base BA; then replay drift and calibration error.
# Reading it: BA 0.50 = coin flip, 1.00 = perfect. AUC is cutoff-free (does the model rank PASS above FAIL?).
# Regression checks are only compared with the base model at the production cutoff; they should not drop.
VARIANTS = [n for n, _ in RUNS if n in RES]
WINNER = max(VARIANTS, key=lambda n: RES[n]["mean_auc_dev_trained"])
print("WINNER (best mean dev AUC over the 5 trained checks):", WINNER, {n: round(RES[n]["mean_auc_dev_trained"], 3) for n in VARIANTS})
print("NOTE: train and dev rows are written and labelled by AI: Claude (stage 3) and, if present (source antigravity-agt), Antigravity. The test sets are Claude-written (unseen industries/personas), NOT independent. Nothing is human-labelled.\n")

for n in VARIANTS:
    R, B = RES[n], RES["base"]
    print(f"=== {n}{'  <- winner' if n == WINNER else ''}")
    print(f"{'check':<18}{'n':>4}{'base BA':>9}{'BA':>7}{'95% interval':>16}{'AUC':>7}{'base AUC':>10}{'tuned cut':>11}")
    for c in TRAINED + REGRESSION:
        r, b = R["checks"][c], B["checks"][c]
        if c in TRAINED:
            lo, hi = r["ba_test_tuned_ci"]
            print(f"{c:<18}{r['n_test']:>4}{b['ba_test_frozen_cut']:>9.3f}{r['ba_test_tuned']:>7.3f}  [{lo:.2f}, {hi:.2f}]{r['auc_test']:>9.3f}{b['auc_test']:>10.3f}{r['tuned_cut']:>11.2f}")
        else:
            lo, hi = r["ba_test_frozen_ci"]
            print(f"{c:<18}{r['n_test']:>4}{b['ba_test_frozen_cut']:>9.3f}{r['ba_test_frozen_cut']:>7.3f}  [{lo:.2f}, {hi:.2f}]{r['auc_test']:>9.3f}{b['auc_test']:>10.3f}{'(frozen)':>11}")
    print("long snippets (padded to real-call length), BA at the same cutoff / AUC, base in brackets: " + "; ".join(
        f"{c} {v['ba']:.2f}/{v['auc']:.2f} ({B['long'][c]['ba']:.2f}/{B['long'][c]['auc']:.2f})" for c, v in R["long"].items()))
    print("natural calls (extractor snippets from 8 complete unseen-industry calls; n is small, intervals wide), BA/AUC, base in brackets: " + "; ".join(
        f"{c} {v['ba']:.2f}/{v['auc']:.2f} n={v['n']} ({B['nat'][c]['ba']:.2f}/{B['nat'][c]['auc']:.2f})" for c, v in R["nat"].items()))
    print(f"mean test BA over trained checks: {R['mean_ba_test_trained']:.3f} | ECE test: {R['ece_test']:.3f} (base {B['ece_test']:.3f})")
    d = R["replay_dev"]
    print(f"replay drift on custom-criteria questions (dev, {d['items']} items): mean |change| {d['mean_abs_diff']:.3f}, "
          f"same verdict as base {d['verdict_agreement'] * 100:.0f}%\n")
print("BA for 'base' uses the production cutoff; for trained checks the tuned model uses a cutoff chosen on DEV, so the two columns are not on identical footing.")

In [ ]:
# CELL 11: save to Drive (temp folder -> verify -> rename), prove the Drive copy loads, print the summary to paste back.
# EXPECT: "copied final-<name>: N files / X MB verified" per variant, "Drive copy loads; max diff ... (must be < 1e-4)",
# then a block starting "=== PASTE THIS BACK TO CLAUDE ===".
from importlib.metadata import version

for n in VARIANTS:
    src = f"{WORK}/ckpt-{n}"
    wait_stable(src, quiet_s=5, ignore=("checkpoint_latest",))
    files, size = copy_verified(src, f"{DRIVE_RUN}/final-{n}", f"final-{n}", ignore=("checkpoint_latest",))
    assert sha256_file(f"{src}/model.safetensors") == sha256_file(f"{DRIVE_RUN}/final-{n}/model.safetensors")
    print(f"copied final-{n}: {files} files / {size / 1e6:.0f} MB verified")

os.makedirs(f"{DRIVE_RUN}/files", exist_ok=True)
for n in VARIANTS:
    for fn in (f"preds_{n}.json", f"train_{n}.log"):
        shutil.copy(f"{WORK}/{fn}", f"{DRIVE_RUN}/files/{fn}")
shutil.copy(f"{WORK}/preds_base.json", f"{DRIVE_RUN}/files/preds_base.json")
for split in ("train_all", "dev_all", "test", "test_long", "test_calls"):
    shutil.copy(f"{DATA_DIR}/{split}.jsonl", f"{DRIVE_RUN}/files/{split}.jsonl")
json.dump(RES, open(f"{DRIVE_RUN}/files/results.json", "w"), indent=1)

manifest = {
    "run_id": RUN_ID, "created": datetime.datetime.now().isoformat(timespec="seconds"), "winner": WINNER,
    "gpu": subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip(),
    "versions": {"python": sys.version.split()[0], "torch": version("torch"), "transformers": version("transformers"),
                 "laya": version("laya"), "laya_commit": LAYA_COMMIT},
    "base": {"repo": BASE_REPO, "revision": BASE_REV, "sha256": BASE_SHA256},
    "frozen_checks_sha256": FROZEN["sha256"], "inputs_sha256": FROZEN["inputs_sha256"],
    "qa_analysis_commit": FROZEN["qa_analysis_commit"], "data_sha256": DATA_SHA256,
    "authorship": "All rows written and labelled by Claude; none human-labelled; test not independent.",
    "training": TRAIN_INFO,
    "final_weights_sha256": {n: sha256_file(f"{DRIVE_RUN}/final-{n}/model.safetensors") for n in VARIANTS},
}
json.dump(manifest, open(f"{DRIVE_RUN}/manifest.json", "w"), indent=2)
os.makedirs(f"{DRIVE}/runs/{RUN_ID}", exist_ok=True)
shutil.copy(f"{DRIVE_RUN}/manifest.json", f"{DRIVE}/runs/{RUN_ID}/manifest.json")
shutil.copy(f"{DRIVE_RUN}/files/results.json", f"{DRIVE}/runs/{RUN_ID}/results.json")

# the Drive copy of the winner must predict what /content predicted
chk = laya.load(f"{DRIVE_RUN}/final-{WINNER}")
sample = DEV[:40]
P = predict_rows(chk, sample)
del chk
torch.cuda.empty_cache()
dmax = max(abs(P[k] - PRED[WINNER]["dev"][k]) for k in P)
print(f"Drive copy loads; max diff vs /content on 40 dev rows: {dmax:.6f}")
assert dmax < 1e-4, "Drive copy predicts differently: stop and tell Claude"

# compact summary
lines = [f"=== PASTE THIS BACK TO CLAUDE ===", f"run {RUN_ID} | winner {WINNER} | gpu {manifest['gpu']}",
         "rows: AI-written and AI-labelled (Claude stage-3 rows; Antigravity AG-T rows if present), nothing human-labelled. test = Claude-written, unseen industries/personas, not independent."]
for n in VARIANTS:
    lines.append(f"[{n}] wall {TRAIN_INFO[n]['wall_s']}s | losses: " + " ".join(l.split("mean loss")[-1].strip().split()[0] for l in TRAIN_INFO[n]["losses"]))
for c in TRAINED + REGRESSION:
    b = RES["base"]["checks"][c]
    parts = [f"{c:<16} base BA {b['ba_test_frozen_cut']:.2f} AUC {b['auc_test']:.2f}"]
    for n in VARIANTS:
        r = RES[n]["checks"][c]
        if c in TRAINED:
            lo, hi = r["ba_test_tuned_ci"]
            parts.append(f"{n} BA {r['ba_test_tuned']:.2f} [{lo:.2f},{hi:.2f}] AUC {r['auc_test']:.2f} cut {r['tuned_cut']:.2f}")
        else:
            parts.append(f"{n} BA {r['ba_test_frozen_cut']:.2f} AUC {r['auc_test']:.2f}")
    lines.append(" | ".join(parts))
for n in VARIANTS:
    lg = RES[n]["long"]
    lines.append(f"[{n}] LONG test BA/AUC: " + "; ".join(f"{c} {v['ba']:.2f}/{v['auc']:.2f}" for c, v in lg.items()) + " | base: " + "; ".join(f"{c} {v['ba']:.2f}/{v['auc']:.2f}" for c, v in RES["base"]["long"].items()))
    lines.append(f"[{n}] NATURAL calls BA/AUC: " + "; ".join(f"{c} {v['ba']:.2f}/{v['auc']:.2f}" for c, v in RES[n]["nat"].items()) + " | base: " + "; ".join(f"{c} {v['ba']:.2f}/{v['auc']:.2f}" for c, v in RES["base"]["nat"].items()))
    d = RES[n]["replay_dev"]
    lines.append(f"[{n}] ECE {RES[n]['ece_test']:.3f} (base {RES['base']['ece_test']:.3f}) | replay same-verdict {d['verdict_agreement'] * 100:.0f}% mean change {d['mean_abs_diff']:.3f}")
lines.append(f"checkpoints on Drive: MyDrive/laya-finetuning/checkpoints/{RUN_ID}/final-{WINNER} (download this folder)")
SUMMARY = "\n".join(lines)
open(f"{DRIVE}/runs/{RUN_ID}/summary.txt", "w").write(SUMMARY + "\n")
print("\n" + SUMMARY)